## TOGSim Configuration
The machine is one TOGSim machine file, named by `TORCHSIM_TARGET`: the compiler compiles for it
and TOGSim times on it. Shipped machines are in `$TORCHSIM_COMPILE_DIR/third_party/TOGSim/configs`.

The GeMM is 4096 x 4096 in bf16: neither operand fits one core's scratchpad beside a tile (the
resident mm declines it), so the template's tile splits M or N and the grid has programs a second
core can run. (At 2048 x 2048 the compiler's plan builds the resident mm instead.) The two
simulations take about 15 minutes.

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_timing_only.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Single Core

In [ ]:
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "single_core")
device = torch.device("npu:0")

input = torch.randn(4096, 4096, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(4096, 4096, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.matmul)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### Multi-Core

In [ ]:
!diff togsim_configs/togsim_config_timing_only.yml togsim_configs/togsim_config_2_cores.yml

In [ ]:
torch._dynamo.reset()
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_2_cores.yml")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "multi_core")

input = torch.randn(4096, 4096, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(4096, 4096, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.matmul)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}